In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# For Market Basket Analysis
from mlxtend.frequent_patterns import apriori, association_rules

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
import warnings
warnings.filterwarnings("ignore")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
file_path = "/content/drive/MyDrive/Retail Analytics System/data/cleaned/clean_retail.csv"

df = pd.read_csv(file_path)

df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,Year,Month,Day
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4,2009,12,1
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009,12,1
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009,12,1
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8,2009,12,1
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0,2009,12,1


In [ ]:
# Keep only required columns for basket analysis

basket_df = df[['Invoice', 'Description', 'Quantity']].copy()

# Remove missing descriptions
basket_df = basket_df.dropna(subset=['Description'])

# Convert invoice and description to string
basket_df['Invoice'] = basket_df['Invoice'].astype(str)
basket_df['Description'] = basket_df['Description'].astype(str)

basket_df.head()

,Invoice,Description,Quantity
0,489434,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12
1,489434,PINK CHERRY LIGHTS,12
2,489434,WHITE CHERRY LIGHTS,12
3,489434,"RECORD FRAME 7"" SINGLE SIZE",48
4,489434,STRAWBERRY CERAMIC TRINKET BOX,24


In [ ]:
# ----------------------------
# Build Basket Matrix
# ----------------------------

basket = (
    basket_df
    .groupby(['Invoice', 'Description'])['Quantity']
    .sum()
    .unstack()
    .fillna(0)
)

# Convert quantities to 0 and 1
basket = basket.applymap(lambda x: 1 if x > 0 else 0)

print("Basket Matrix Shape:", basket.shape)

basket.head()

Basket Matrix Shape: (36969, 5283)


Description,DOORMAT UNION JACK GUNS AND ROSES,3 STRIPEY MICE FELTCRAFT,4 PURPLE FLOCK DINNER CANDLES,50'S CHRISTMAS GIFT BAG LARGE,ANIMAL STICKERS,BLACK PIRATE TREASURE CHEST,BROWN PIRATE TREASURE CHEST,Bank Charges,CAMPHOR WOOD PORTOBELLO MUSHROOM,CHERRY BLOSSOM DECORATIVE FLASK,DOLLY GIRL BEAKER,FAIRY CAKE CANDLES,FLAMINGO LIGHTS,HOME SWEET HOME BLACKBOARD,I LOVE LONDON MINI BACKPACK,I LOVE LONDON MINI RUCKSACK,IVORY PAPER CUP CAKE CASES,LARGE SKULL WINDMILL,NEW BAROQUE BLACK BOXES,NINE DRAWER OFFICE TIDY,OVAL WALL MIRROR DIAMANTE,PAINT YOUR OWN CANVAS SET,PEACE WOODEN BLOCK LETTERS,RED SPOT GIFT BAG LARGE,RED/WHITE DOT MINI CASES,RIDGED GLASS T-LIGHT HOLDER,SET 2 TEA TOWELS I LOVE LONDON,SET Of 6 SOLDIER SKITTLES,SILVER CHERRY LIGHTS,SILVER T-LIGHT SETTING,SPACEBOY BABY GIFT SET,STAR T-LIGHT HOLDER,TOADSTOOL BEDSIDE LIGHT,TRELLIS COAT RACK,VINTAGE DESIGN GIFT TAGS,WHITE BAMBOO RIBS LAMPSHADE,WHITE CHERRY LIGHTS,10 COLOUR SPACEBOY PEN,11 PC CERAMIC TEA SET POLKADOT,12 ASS ZINC CHRISTMAS DECORATIONS,12 COLOURED PARTY BALLOONS,12 DAISY PEGS IN WOOD BOX,12 EGG HOUSE PAINTED WOOD,12 HANGING EGGS HAND PAINTED,12 IVORY ROSE PEG PLACE SETTINGS,12 MESSAGE CARDS WITH ENVELOPES,12 MINI TOADSTOOL PEGS,12 PENCIL SMALL TUBE WOODLAND,12 PENCILS SMALL TUBE POSY,12 PENCILS SMALL TUBE RED RETROSPOT,12 PENCILS SMALL TUBE RED SPOTTY,12 PENCILS SMALL TUBE SKULL,12 PENCILS TALL TUBE POSY,12 PENCILS TALL TUBE RED RETROSPOT,12 PENCILS TALL TUBE RED SPOTTY,12 PENCILS TALL TUBE SKULLS,12 PENCILS TALL TUBE WOODLAND,12 PINK HEN+CHICKS IN BASKET,12 PINK ROSE PEG PLACE SETTINGS,12 RED ROSE PEG PLACE SETTINGS,15 PINK FLUFFY CHICKS IN BOX,15CM CHRISTMAS GLASS BALL 20 LIGHTS,16 PC CUTLERY SET PANTRY DESIGN,16 PIECE CUTLERY SET PANTRY DESIGN,18PC WOODEN CUTLERY SET DISPOSABLE,2 DAISIES HAIR COMB,2 PICTURE BOOK EGGS EASTER BUNNY,2 PICTURE BOOK EGGS EASTER CHICKS,2 PICTURE BOOK EGGS EASTER DUCKS,20 DOLLY PEGS RETROSPOT,200 BENDY SKULL STRAWS,200 RED + WHITE BENDY STRAWS,24 HANGING EASTER EGGS FLORAL TUB,3 BIRDS CANVAS SCREEN,3 BLACK CATS W HEARTS BLANK CARD,3 DRAWER ANTIQUE WHITE WOOD CABINET,3 GARDENIA MORRIS BOXED CANDLES,3 HEARTS HANGING DECORATION RUSTIC,3 HOOK HANGER MAGIC GARDEN,3 HOOK PHOTO SHELF ANTIQUE WHITE,3 PIECE JIGSAW TOADSTOOLS,3 PIECE SPACEBOY COOKIE CUTTER SET,3 PINK HEN+CHICKS IN BASKET,3 RAFFIA RIBBONS 50'S CHRISTMAS,3 RAFFIA RIBBONS VINTAGE CHRISTMAS,3 ROSE MORRIS BOXED CANDLES,3 STRIPEY MICE FELTCRAFT,3 TIER CAKE TIN GREEN AND CREAM,3 TIER CAKE TIN RED AND CREAM,3 TIER SWEETHEART GARDEN SHELF,3 TRADITIONAL COOKIE CUTTERS SET,3 TRADITIONAl BISCUIT CUTTERS SET,3 WHITE CHOC MORRIS BOXED CANDLES,3 WICK CHRISTMAS BRIAR CANDLE,36 DOILIES DOLLY GIRL,36 DOILIES SPACEBOY DESIGN,36 DOILIES VINTAGE CHRISTMAS,36 FOIL HEART CAKE CASES,36 FOIL STAR CAKE CASES,36 PENCILS TUBE POSY,36 PENCILS TUBE RED RETROSPOT,36 PENCILS TUBE RED SPOTTY,36 PENCILS TUBE SKULLS,36 PENCILS TUBE WOODLAND,3D CHRISTMAS STAMPS STICKERS,3D DOG PICTURE PLAYING CARDS,3D HEARTS HONEYCOMB PAPER GARLAND,3D SHEET OF CAT STICKERS,3D SHEET OF DOG STICKERS,3D SHEET OF SEA WORLD STICKERS,3D STICKERS CHRISTMAS STAMPS,3D STICKERS TRADITIONAL CHRISTMAS,3D STICKERS VINTAGE CHRISTMAS,3D TRADITIONAL CHRISTMAS STICKERS,3D VINTAGE CHRISTMAS STICKERS,4 BLUE DINNER CANDLES SILVER FLOCK,4 BURGUNDY WINE DINNER CANDLES,4 FESTIVE GREEN DINNER CANDLES,4 GOLD FLOCK CHRISTMAS BALLS,4 IVORY DINNER CANDLES GOLD FLOCK,4 IVORY DINNER CANDLES SILVER FLOCK,4 LAVENDER BOTANICAL DINNER CANDLES,4 LILY BOTANICAL DINNER CANDLES,4 PEAR BOTANICAL DINNER CANDLES,4 PINK DINNER CANDLE SILVER FLOCK,4 PINK FLOCK CHRISTMAS BALLS,4 ROSE PINK DINNER CANDLES,4 SKY BLUE DINNER CANDLES,4 TRADITIONAL SPINNING TOPS,4 VANILLA BOTANICAL CANDLES,4 WILDFLOWER BOTANICAL CANDLES,5 HOOK BLACKBOARD ORGANISER,5 HOOK HANGER MAGIC TOADSTOOL,5 HOOK HANGER RED MAGIC TOADSTOOL,5 STRAND GLASS NECKLACE AMBER,5 STRAND GLASS NECKLACE AMETHYST,5 STRAND GLASS NECKLACE BLACK,5 STRAND GLASS NECKLACE CRYSTAL,50'S CHRISTMAS PAPER GIFT BAG,50CM METAL STR

In [7]:
# ----------------------------
# Run Apriori Algorithm
# ----------------------------

frequent_itemsets = apriori(
    basket,
    min_support=0.02,
    use_colnames=True
)

print("Number of Frequent Itemsets:", len(frequent_itemsets))

frequent_itemsets.sort_values(
    by='support',
    ascending=False
).head(20)

Number of Frequent Itemsets: 173


,support,itemsets
153,0.132219,(WHITE HANGING HEART T-LIGHT HOLDER)
122,0.089724,(REGENCY CAKESTAND 3 TIER)
8,0.071736,(ASSORTED COLOUR BIRD ORNAMENT)
61,0.070654,(JUMBO BAG RED RETROSPOT)
96,0.056182,(PARTY BUNTING)
72,0.054018,(LUNCH BAG BLACK SKULL.)
79,0.050691,(LUNCH BAG SPACEBOY DESIGN )
125,0.050231,(REX CASH+CARRY JUMBO SHOPPER)
49,0.049528,(HOME BUILDING BLOCK WORD)
145,0.049176,(STRAWBERRY CERAMIC TRINKET BOX)


In [8]:
# ----------------------------
# Generate Association Rules
# ----------------------------

rules = association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=1
)

# Sort by highest lift
rules = rules.sort_values(
    by='lift',
    ascending=False
)

print("Total Association Rules:", len(rules))

rules.head(20)

Total Association Rules: 16


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(ROSES REGENCY TEACUP AND SAUCER ),(GREEN REGENCY TEACUP AND SAUCER),0.028564,0.025237,0.020098,0.703598,27.879242,1.0,0.019377,3.288656,0.992481,0.596308,0.695924,0.749977
1,(GREEN REGENCY TEACUP AND SAUCER),(ROSES REGENCY TEACUP AND SAUCER ),0.025237,0.028564,0.020098,0.796356,27.879242,1.0,0.019377,4.770260,0.989093,0.596308,0.790368,0.749977
13,(SWEETHEART CERAMIC TRINKET BOX),(STRAWBERRY CERAMIC TRINKET BOX),0.031675,0.049176,0.022100,0.697694,14.187602,1.0,0.020542,3.145239,0.959922,0.376151,0.682059,0.573545
12,(STRAWBERRY CERAMIC TRINKET BOX),(SWEETHEART CERAMIC TRINKET BOX),0.049176,0.031675,0.022100,0.449395,14.187602,1.0,0.020542,1.758656,0.977590,0.376151,0.431384,0.573545
14,(WOODEN PICTURE FRAME WHITE FINISH),(WOODEN FRAME ANTIQUE WHITE ),0.044848,0.047959,0.026860,0.598914,12.488023,1.0,0.024709,2.373660,0.963118,0.407301,0.578710,0.579491
15,(WOODEN FRAME ANTIQUE WHITE ),(WOODEN PICTURE FRAME WHITE FINISH),0.047959,0.044848,0.026860,0.560068,12.488023,1.0,0.024709,2.171133,0.966264,0.407301,0.539411,0.579491
5,(LOVE BUILDING BLOCK WORD),(HOME BUILDING BLOCK WORD),0.040169,0.049528,0.021261,0.529293,10.686745,1.0,0.019272,2.019243,0.944360,0.310672,0.504765,0.479283
4,(HOME BUILDING BLOCK WORD),(LOVE BUILDING BLOCK WORD),0.049528,0.040169,0.021261,0.429274,10.686745,1.0,0.019272,1.681771,0.953659,0.310672,0.405389,0.479283
2,(HEART OF WICKER SMALL),(HEART OF WICKER LARGE),0.048121,0.047148,0.023506,0.488477,10.360582,1.0,0.021237,1.862774,0.949155,0.327554,0.463166,0.493521
3,(HEART OF WICKER LARGE),(HEART OF WICKER SMALL),0.047148,0.048121,0.023506,0.498566,10.360582,1.0,0.021237,1.898312,0.948185,0.327554,0.473216,0.493521


In [9]:
# ----------------------------
# Extract Top 20 Product Pairs
# ----------------------------

top_rules = rules[[
    'antecedents',
    'consequents',
    'support',
    'confidence',
    'lift'
]].head(20)

top_rules

,antecedents,consequents,support,confidence,lift
0,(ROSES REGENCY TEACUP AND SAUCER ),(GREEN REGENCY TEACUP AND SAUCER),0.020098,0.703598,27.879242
1,(GREEN REGENCY TEACUP AND SAUCER),(ROSES REGENCY TEACUP AND SAUCER ),0.020098,0.796356,27.879242
13,(SWEETHEART CERAMIC TRINKET BOX),(STRAWBERRY CERAMIC TRINKET BOX),0.022100,0.697694,14.187602
12,(STRAWBERRY CERAMIC TRINKET BOX),(SWEETHEART CERAMIC TRINKET BOX),0.022100,0.449395,14.187602
14,(WOODEN PICTURE FRAME WHITE FINISH),(WOODEN FRAME ANTIQUE WHITE ),0.026860,0.598914,12.488023
15,(WOODEN FRAME ANTIQUE WHITE ),(WOODEN PICTURE FRAME WHITE FINISH),0.026860,0.560068,12.488023
5,(LOVE BUILDING BLOCK WORD),(HOME BUILDING BLOCK WORD),0.021261,0.529293,10.686745
4,(HOME BUILDING BLOCK WORD),(LOVE BUILDING BLOCK WORD),0.021261,0.429274,10.686745
2,(HEART OF WICKER SMALL),(HEART OF WICKER LARGE),0.023506,0.488477,10.360582
3,(HEART OF WICKER LARGE),(HEART OF WICKER SMALL),0.023506,0.498566,10.360582


In [10]:
# ---------------------------------------
# Print Top 20 Product Associations
# ---------------------------------------

print("Top 20 Product Associations\n")

for i, row in top_rules.iterrows():

    product_a = ', '.join(list(row['antecedents']))
    product_b = ', '.join(list(row['consequents']))

    print(f"{product_a}  ➜  {product_b}")
    print(f"Support    : {row['support']:.4f}")
    print(f"Confidence : {row['confidence']:.4f}")
    print(f"Lift       : {row['lift']:.4f}")
    print("-" * 60)

Top 20 Product Associations

ROSES REGENCY TEACUP AND SAUCER   ➜  GREEN REGENCY TEACUP AND SAUCER
Support    : 0.0201
Confidence : 0.7036
Lift       : 27.8792
------------------------------------------------------------
GREEN REGENCY TEACUP AND SAUCER  ➜  ROSES REGENCY TEACUP AND SAUCER 
Support    : 0.0201
Confidence : 0.7964
Lift       : 27.8792
------------------------------------------------------------
SWEETHEART CERAMIC TRINKET BOX  ➜  STRAWBERRY CERAMIC TRINKET BOX
Support    : 0.0221
Confidence : 0.6977
Lift       : 14.1876
------------------------------------------------------------
STRAWBERRY CERAMIC TRINKET BOX  ➜  SWEETHEART CERAMIC TRINKET BOX
Support    : 0.0221
Confidence : 0.4494
Lift       : 14.1876
------------------------------------------------------------
WOODEN PICTURE FRAME WHITE FINISH  ➜  WOODEN FRAME ANTIQUE WHITE 
Support    : 0.0269
Confidence : 0.5989
Lift       : 12.4880
------------------------------------------------------------
WOODEN FRAME ANTIQUE WHIT

In [11]:
# ---------------------------------------
# Export Basket Rules
# ---------------------------------------

output_path = "/content/drive/MyDrive/Retail Analytics System/data/cleaned/basket_rules.csv"

top_rules.to_csv(output_path, index=False)

print("Basket Rules exported successfully!")

Basket Rules exported successfully!
